In [ ]:
!pip install yt-dlp openai-whisper transformers torch

import yt_dlp
import whisper
import torch
import os
import glob
from google.colab import files
from transformers import pipeline

ERROR: Could not find a version that satisfies the requirement intel-extension-for-pytorch (from versions: none)

[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for intel-extension-for-pytorch


Defaulting to user installation because normal site-packages is not writeable


ModuleNotFoundError: No module named 'google.colab'

In [ ]:
VIDEO_URL = "https://www.youtube.com/watch?v=TLKxdTmk-zc"
def download_audio(url):
    print("--- Starting Download ---")
    ydl_opts = {
        'format': 'bestaudio/best',
        'postprocessors': [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'mp3',
            'preferredquality': '192',
        }],
        'outtmpl': 'audio_file.%(ext)s',
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([url])
    return "audio_file.mp3"

def transcribe_audio(file_path):
    model = whisper.load_model("base")

    result = model.transcribe(file_path)
    text = result["text"]

    with open("transcript.txt", "w", encoding="utf-8") as f:
        f.write(text)
    return text

def summarize_transcript(text):
    print("--- Loading BART Summarizer (Manual Load) ---")

    from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
    import torch

    model_name = "facebook/bart-large-cnn"
    device = "cuda" if torch.cuda.is_available() else "cpu"

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

    words = text.split()
    chunks = [" ".join(words[i : i + 500]) for i in range(0, len(words), 500)]

    full_summary = []

    for i, chunk in enumerate(chunks):
        if len(chunk.split()) < 30: continue

        inputs = tokenizer(chunk, return_tensors="pt", max_length=1024, truncation=True).to(device)
        summary_ids = model.generate(
            inputs["input_ids"],
            max_length=150,
            min_length=40,
            length_penalty=2.0,
            num_beams=4,
            early_stopping=True
        )

        result = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
        full_summary.append(result)

    return " ".join(full_summary)

try:
    audio_file = download_audio(VIDEO_URL)

    transcript = transcribe_audio(audio_file)
    print(transcript[:200] + "...")

    final_summary = summarize_transcript(transcript)

    print("\n" + "="*40)
    print("FINAL SUMMARY")
    print("="*40)
    print(final_summary)

except Exception as e:
    print(f"An error occurred: {e}")

--- Starting Download ---
[youtube] Extracting URL: https://www.youtube.com/watch?v=TLKxdTmk-zc
[youtube] TLKxdTmk-zc: Downloading webpage


[youtube] TLKxdTmk-zc: Downloading android vr player API JSON
[info] TLKxdTmk-zc: Downloading 1 format(s): 251
[download] Destination: audio_file.webm
[download] 100% of    9.03MiB in 00:00:00 at 25.89MiB/s  
[ExtractAudio] Destination: audio_file.mp3
Deleting original file audio_file.webm (pass -k to keep)


100%|███████████████████████████████████████| 139M/139M [00:01<00:00, 95.2MiB/s]


 Your brain is the most powerful weapon in the world. Once you put away your phones and your computers and all that, we have nowadays, that's great. We're up to date. But your brain is the only thing ...
--- Loading BART Summarizer (Manual Load) ---


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


...

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]


FINAL SUMMARY
"There's 24 hours in the day where you're alone in this brain," he says. "If you can't control your own brain and your brain controls you, you're... You got to tell your brain where you want to go" I became hell. That became my new norm. I gave myself no way out. There was nothing outside these walls of hell. I became, I love God. But for a short period of time, I became the devil. That was my mindset. And that's how you get through things. You put yourself, you immerse yourself wherever it is, you become that. "When you know that you can run on broken legs, and you can do certain things that a lot of people can do, but they're not willing to do," he says. "The feeling you get is basically invincibility. When you find your true passion in life, am I passionate for me when I want to be now?" was so disappointed in what I saw every day. I wanted everybody to love David Goggins. I didn't love myself. But I knew a lot of us wanna find peace first. No one really finds himself

In [ ]:
! pip install deep_translator gtts

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 9.4 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.3.2
    Uninstalling click-8.3.2:
      Successfully uninstalled click-8.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
typer 0.24.1 requires click>=8.2.1, but you have click 8.1.8 which is incompatible.


In [ ]:


from deep_translator import GoogleTranslator
from gtts import gTTS
from IPython.display import Audio, display

def translate_and_read(english_text, target_lang='hi'):
    # Step A: Translate the English summary to the target language
    print(f"Translating summary to {target_lang}...")
    translated_text = GoogleTranslator(source='en', target=target_lang).translate(english_text)

    print(f"Translated Text Preview: {translated_text[:100]}...")

    print("Generating audio...")
    tts = gTTS(text=translated_text, lang=target_lang)
    tts.save("multilingual_summary.mp3")

    display(Audio("multilingual_summary.mp3", autoplay=False))

print("In which language would you like to hear the summary?")
print("1.  English (en)")
print("2.  Kannada (kn)")
print("3.  Hindi (hi)")
print("4.  Tamil (ta)")
print("5.  Telugu (te)")
print("6.  Malayalam (ml)")
print("7.  Marathi (mr)")
print("8.  Bengali (bn)")
print("9.  Gujarati (gu)")
print("10. Spanish (es)")
print("11. French (fr)")
print("12. Japanese (ja)")
target=input("")
translate_and_read(final_summary, target_lang=target)

In which language would you like to hear the summary?
1.  English (en)
2.  Kannada (kn)
3.  Hindi (hi)
4.  Tamil (ta)
5.  Telugu (te)
6.  Malayalam (ml)
7.  Marathi (mr)
8.  Bengali (bn)
9.  Gujarati (gu)
10. Spanish (es)
11. French (fr)
12. Japanese (ja)
kn
Translating summary to kn...
Translated Text Preview: "ಈ ಮೆದುಳಿನಲ್ಲಿ ನೀವು ಒಬ್ಬಂಟಿಯಾಗಿರುವ ದಿನದ 24 ಗಂಟೆಗಳು" ಎಂದು ಅವರು ಹೇಳುತ್ತಾರೆ. "ನಿಮ್ಮ ಮೆದುಳನ್ನು ನಿಯಂತ್ರಿಸ...
Generating audio...
